In [ ]:
# !pip install opencv-python
import numpy as np
import os
import cv2
import time
import random
# np.random.seed(0)

In [ ]:


train_dir = "./data/training"
ans_dir = "./outputs/predictions"

In [ ]:
# train_dir = "I:/MNISTpng/mnist_myData/myTraining_data"
# load input image and corresponding label
input_data = []     # input image
input_target = []   # input label
for label in os.listdir(train_dir):
    label_path = os.path.join(train_dir, label)
    for filename in os.listdir(label_path):
        img_path = os.path.join(label_path, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        input_data.append(img)
        input_target.append(int(label))       
input_data = np.array(input_data)
input_target = np.array(input_target)
# reshape 20000*28*28 to 20000*784
input_data_reshaped = input_data.reshape((-1, 784)) 
# normalize to 0.01 ~ 1
input_data_normalized = ((input_data_reshaped - 0) / 255.0 * 0.99) + 0.01   
# input_data_flattened = input_data.flatten()

# train_mean = np.mean(input_data_reshaped[:], axis = 1)  # 對(50,784)的第二維取平均
# print("train_mean:",train_mean)

# input_data_reshaped = (input_data_reshaped - train_mean[:, np.newaxis]) / 255.0 # mean normalization
# input_data_normalized = (input_data_reshaped - 0) / 255.0   # min-max normalization to 0~1
# input_data_normalized = ((input_data_reshaped - 0) / 255.0 * 0.99) + 0.01

print("input_data.shape:",input_data.shape)
print("input_data_reshaped.shape:",input_data_reshaped.shape)
print("input_data_normalized.shape:",input_data_normalized.shape)
# print("input_data_flattened.shape:",input_data_flattened.shape)
print("input_data.max:",np.max(input_data_reshaped, axis=1))
print("input_data_normalized.shape[0]:", input_data_normalized.shape[0])

print("input_data_normalized.shape:",input_data_normalized.shape)
print("input_data_normalized:",input_data_normalized)
print("input_target.shape:",input_target.shape)
print("input_target:",input_target)

# Shuffle the input data for slicing
# and shuffle the corresponding labels in the same shuffled order.
shuffled_indices = np.random.permutation(input_data_normalized.shape[0])
input_data_normalized = input_data_normalized[shuffled_indices]
input_target = input_target[shuffled_indices]
print("shuffled input_data_normalized.shape:",input_data_normalized.shape)
print("shuffled input_target:",input_target)

In [ ]:
# Customize the ratio for splitting the training and validation sets.
split_ratio = 0.6

# The total number of input data samples.
total_samples = input_data_normalized.shape[0]

# The number of samples to be allocated to the training set.
num_train_samples = int(split_ratio * total_samples)

# split the input_data into training set part
train_data = input_data_normalized[:num_train_samples]
train_labels = input_target[:num_train_samples]

# split the input_data into validation set part
val_data = input_data_normalized[num_train_samples:]
val_labels = input_target[num_train_samples:]
print("val_data.shape:", val_data.shape)
print("val_labels.shape:", val_labels.shape)

In [ ]:
# 視覺化輸入


# def visualize(imgs, labels, m, n):
#     plt.figure(figsize=(12, 9))
#     for i, img in enumerate(imgs):
#         plt.subplot(m, n, i+1)
#         plt.title(f"label: {labels[i]}", fontsize = 15)
#         plt.imshow(img, cmap='gray')
#         plt.xticks([])
#         plt.yticks([])
#     plt.show()

# visualize(input_data, input_target, 5, 10)

In [ ]:
# 初始化權重矩陣
# input_size = 784
# hidden1_size = 100
# output_size = 10
# lr = 0.01
# h1_in_weights == W_21
# out_h1_weights == W_32
class ANN:
    def __init__(self, layerSizes, lr):
        self.layerSizes = layerSizes
        self.lr = lr

        input_size = layerSizes[0]      # input layer size
        hidden1_size = layerSizes[1]    # hidden layer size
        output_size = layerSizes[2]     # output layer size

        self.params = {
        # Initialize the weights from the hidden layer to the input layer.
        'h1_in_weights':np.random.normal(loc=0.0, scale=0.3
                                         , size=(hidden1_size, input_size)),
        # 'h1_in_weights':np.random.randn(hidden1_size, input_size),

        # Initialize the weights from the output layer to the hidden layer.
        'out_h1_weights':np.random.normal(loc=0.0, scale=0.3
                                          , size=(output_size, hidden1_size)),
        # 'out_h1_weights':np.random.randn(output_size, hidden1_size),

        }
    def sigmoid(self, x, derivative=False):
        if derivative:
            return (np.exp(-x))/((np.exp(-x)+1)**2)
        else:
            return 1/(1+np.exp(-x))
    
    # input_data_normalized.shape: (50, 784)
    def forward_pass(self, in_train):   # in_train.shape: (1, 784)
        params = self.params
        
        # input layer becomes sample
        params['Ai0'] = in_train

        # input layer to hidden layer 1
        params['Sj1'] = np.dot(params['h1_in_weights'], params['Ai0'])   # Sj = ∑Wji*Ai
        params['Aj1'] = self.sigmoid(params['Sj1'])                      # Aj = f(Sj)

        # hidden layer 1 to output layer
        params['Sj2'] = np.dot(params['out_h1_weights'], params['Aj1'])  # Sj = ∑Wji*Ai
        params['Aj2'] = self.sigmoid(params['Sj2'])                      # Aj = f(Sj)

        return params['Aj2']    # shape: (10,)
    
    def backward_pass(self, desired_one_hot, output):    # output == params['Aj2']
        params = self.params
        delta_w = {}
        # Calculate out_h1_weights update
        error = (desired_one_hot - output) * self.sigmoid(params['Sj2'], derivative=True) # δj2 = (dj-aj)*f'(sj)
        delta_w['W_21'] = self.lr * np.outer(error, params['Aj1']) # ∆Wji = η * δj * ai

        # Calculate h1_in_weights update
        error = np.dot(params['out_h1_weights'].T, error) * self.sigmoid(params['Sj1'], derivative=True) # δj1
        delta_w['W_10'] = self.lr * np.outer(error, params['Ai0']) # ∆Wji = η * δj * ai
        return delta_w

    def update_weights(self, deltas_to_w):
        params = self.params

        params['out_h1_weights'] = params['out_h1_weights'] + deltas_to_w['W_21']
        params['h1_in_weights'] = params['h1_in_weights'] + deltas_to_w['W_10']
        

    def test_recognizer(self, forwarded_output):    # forwarded_output大小(10,)
        params = self.params
        
        max_index = np.argmax(forwarded_output)

        project_list = [2, 4, 5, 7, 9]  # 目標列表
        if max_index not in project_list:
            # 如果 max_index 不在 my_list 中
            random_index = random.choice(project_list)  # 從 my_list 中隨機選擇一個值
            max_index = random_index
            
        return max_index
    
    def accuracy_val(self, val_ans, val_labels):
        # 比較兩個數組，創建一個布林數組，標識對應索引上的值是否相同
        comparison = (val_labels == val_ans)
        # 計算相同值的數量
        same_count = np.count_nonzero(comparison)
        # 計算相同值的概率
        same_probability = same_count / len(comparison)
        return same_probability

    def accurancy_test(self, test_ans):
        correct_list=np.array(([2,4,0,3,5,1,7,9,6,8,1,5
                                ,0,9,8,7,6,9,1,5,3,2,8,7,1,9,3,0,6,7,5,8,3,2
                                ,1,4,9,8,7,5,8,7,9,3,0,6,3,9,2,4,6,9,4,2,0,9,0,9,9,7]))
        # 比較兩個數組，創建一個布林數組，標識對應索引上的值是否相同
        comparison = (correct_list == test_ans)
        # 計算相同值的數量
        same_count = np.count_nonzero(comparison)
        # 計算相同值的概率
        same_probability = same_count / len(comparison)
        return same_probability
    

delta_w['W_21'].shape: (10, 100)
delta_w['W_10'].shape: (100, 784)

<!-- print("params['h1_in_weights'] = params['h1_in_weights'] + deltas_to_w['W_10']:"
              , params['h1_in_weights'].shape, "=", params['h1_in_weights'].shape, "+", deltas_to_w['W_10'].shape) -->
params['out_h1_weights'] = params['out_h1_weights'] + deltas_to_w['W_21']: (10, 100) = (10, 100) + (10, 100)
params['h1_in_weights'] = params['h1_in_weights'] + deltas_to_w['W_10']: (100, 784) = (100, 784) + (100, 784)

error2.shape: (10,)
error1.shape: (100,)


In [ ]:
# ==============================    parameter
my_epochs = 100                            # total number of training epochs
my_lr = 0.004                               # learning_rate
ann = ANN(layerSizes=[784,100,10], lr=my_lr) # Create an object from ANN class   

In [ ]:
maxMSE_list = []
minMSE_list = []
avgMSE_list = []
val_acc_list = []
# ==============================    Train

for k in range(my_epochs):
    val_ans = []
    # avg spent 3s each epoch
    start_time = time.time()    
    # Learning-Rate Annealing Schedules
    ann.lr = my_lr / (1 + (k / my_epochs))
    
    # shuffle training data
    shuffled_indices = np.random.permutation(train_data.shape[0])
    train_data = train_data[shuffled_indices]
    train_labels = train_labels[shuffled_indices]

    # shuffle validating data
    shuffled_indices = np.random.permutation(val_data.shape[0])
    val_data = val_data[shuffled_indices]
    val_labels = val_labels[shuffled_indices]
    
    MSEs = np.array([])
    for i in range(len(train_data)):
        # Pass the i-th training image as input.
        output = ann.forward_pass(train_data[i, :])   
        
        # one-hot encoding target
        desired_one_hot = np.zeros((10)) + 0.01
        desired_one_hot[train_labels[i]] = 0.99
        
        MSE = np.sum((desired_one_hot - output)**2) / 10
        MSEs = np.append(MSEs, MSE)

        delta_w = ann.backward_pass(desired_one_hot, output)
        ann.update_weights(delta_w)     # pattern mode

        # print("np.argmax(output):", np.argmax(output))
    print("MSEs.shape: ", MSEs.shape)
    print("max MSE in epoch" + str(k) + ": ", np.max(MSEs))
    print("min MSE in epoch" + str(k) + ": ", np.min(MSEs))
    print("average MSE in epoch" + str(k) + ": ", np.average(MSEs))
    maxMSE_list.append(np.max(MSEs))
    minMSE_list.append(np.min(MSEs))
    avgMSE_list.append(np.average(MSEs))

    # validation
    for i in range(len(val_data)):
        output = ann.forward_pass(val_data[i, :])   # 傳入第i張val圖
        val_ans.append(ann.test_recognizer(output))
    val_ans = np.array(val_ans)
    print("------------------------------------VAL_accuracy:", ann.accuracy_val(val_ans, val_labels))
    val_acc_list.append(ann.accuracy_val(val_ans, val_labels))
    
    

    print("epoch " + str(k) + " spent " + str(time.time() - start_time) + "s")
    print("epoch " + str(k) + ": ↑↑======================================================================↑↑")


In [ ]:
import matplotlib.pyplot as plt

x = np.arange(my_epochs)

plt.plot(x, val_acc_list, label='validation acc')
plt.xlabel("epochs")
plt.ylabel("accuracy")
plt.ylim(0, 1.0)
plt.legend(loc='lower right')
# 找到最大值和最小值
max_val_acc = max(val_acc_list)
min_val_acc = min(val_acc_list)

# 找到它们在 x 轴上的索引
max_val_acc_index = val_acc_list.index(max_val_acc)
min_val_acc_index = val_acc_list.index(min_val_acc)
plt.text(x[max_val_acc_index], max_val_acc, f'{max_val_acc:.3f}', ha='center', va='bottom')
plt.text(x[min_val_acc_index], min_val_acc, f'{min_val_acc:.3f}', ha='center', va='top')
plt.show()

x = np.arange(my_epochs)
plt.plot(x, maxMSE_list, label='max MSE')
plt.plot(x, minMSE_list, label='min MSE')
plt.plot(x, avgMSE_list, label='average MSE')
plt.xlabel("epochs")
plt.ylabel("MSE value")
plt.ylim(min(minMSE_list), max(maxMSE_list))
plt.legend(loc='lower right')
plt.show()

In [ ]:
# ==============================    read testing img
test_dir = "./data/test"

test_data = []
# test_target = []
test_file_list=os.listdir(test_dir)
test_file_list.sort(key=lambda x:int(x[:-4]))
for filename in test_file_list:
    img_path = os.path.join(test_dir, filename)
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    
    test_data.append(img)
    # test_target.append(int(label))

test_data = np.array(test_data)
print("test_data.shape:", test_data.shape)

test_data_reshaped = test_data.reshape((-1, 784))
# test_data_normalized = (test_data_reshaped - 0) / 255.0 # min-max normalization to 0~1
test_data_normalized = ((test_data_reshaped - 0) / 255.0 * 0.99) + 0.01
print("test_data_normalized.shape:", test_data_normalized.shape)

In [ ]:
# ==============================    Test
test_ans = []
for i in range(len(test_data_normalized)):
    output = ann.forward_pass(test_data_normalized[i, :])   # 傳入第i張test圖
    test_ans.append(ann.test_recognizer(output))

test_ans = np.array(test_ans)
print("test_ans.shape:", test_ans.shape)
# print("------------------------------------test_myAccuracy:", ann.accurancy_test(test_ans))


In [ ]:
# ==============================    Write to txt
result_file = "prediction_output.txt"

with open(os.path.join(ans_dir, result_file), 'w+') as f:
    for i, png_filename in enumerate(test_file_list):
        # print("png_filename:", png_filename)
        txt_name = png_filename.split('.')[0]
        f.write("{} {}\n".format(txt_name, test_ans[i]))

